# GPT-light Kaggle Notebook

Lädt einen kleinen Chat-Datensatz, erstellt ein einfaches GPT-Modell und trainiert es auf einer kleinen Menge Daten. Portiert aus dem Colab-Notebook, angepasst für Kaggle (GPU aktiviert, Internet aktiviert, kein interaktiver input()).

In [ ]:
!pip install -q transformers

In [ ]:
import time
import glob
import os
import json
import numpy as np
import torch
from transformers import PreTrainedTokenizerFast
import torch.nn as nn
from torch.nn import functional as F

device = 'cpu'
if torch.cuda.is_available():
    try:
        torch.zeros(1, device='cuda') + 1
        device = 'cuda'
    except RuntimeError as e:
        print('GPU found but unusable with this PyTorch build, falling back to CPU:', e)
print('device:', device)
print('GPU count:', torch.cuda.device_count())

if device == 'cuda':
    torch.backends.cudnn.benchmark = True

In [ ]:
# Phase 3: pretokenized fineweb-edu corpus (prepared offline, uploaded as a
# Kaggle dataset) replaces the runtime download+tokenize of smol-smoltalk.
# This also means training sessions no longer burn GPU-quota time on data prep.
data_bin_candidates = glob.glob('/kaggle/input/*/train.bin')
assert data_bin_candidates, (
    'Pretokenized dataset not found. Attach the fineweb-edu-16k dataset '
    'as a dataset_source in kernel-metadata.json before pushing.'
)
data_dir = os.path.dirname(data_bin_candidates[0])
print('Using pretokenized data from', data_dir)

with open(os.path.join(data_dir, 'meta.json')) as f:
    meta = json.load(f)
vocab_size = meta['vocab_size']

train_data = np.memmap(os.path.join(data_dir, 'train.bin'), dtype=np.uint16, mode='r')
val_data = np.memmap(os.path.join(data_dir, 'val.bin'), dtype=np.uint16, mode='r')

tokenizer = PreTrainedTokenizerFast.from_pretrained(data_dir)
tokenizer.pad_token = '<|endoftext|>'
tokenizer.eos_token = '<|endoftext|>'

print('meta:', meta)
print('train tokens:', len(train_data))
print('val tokens:', len(val_data))
print('vocab_size:', vocab_size)
print(tokenizer.decode(train_data[:100].astype(np.int64).tolist()))

In [ ]:
torch.manual_seed(1337)

# batch_size deliberately conservative: at n_embd=768 / 12 layers / block_size=512
# the per-GPU activation memory is ~4-5x that of the Phase 2 smoke test, so 48
# would risk OOM on a 16GB T4. grad_accum_steps (below) is raised to keep the
# effective batch (sequences per optimizer step) unchanged.
batch_size = 24
block_size = 512

def get_batch(split):
    data_source = train_data if split == 'train' else val_data
    ix = np.random.randint(0, len(data_source) - block_size, size=(batch_size,))
    x = torch.from_numpy(np.stack([data_source[i:i+block_size].astype(np.int64) for i in ix]))
    y = torch.from_numpy(np.stack([data_source[i+1:i+block_size+1].astype(np.int64) for i in ix]))
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print(xb.shape, yb.shape)
print(xb.device, yb.device)

In [ ]:
n_embd = 768
n_head = 12
n_layer = 12
dropout = 0.1
learning_rate = 3e-4
max_iters = 4000
eval_interval = 250
eval_iters = 50
grad_accum_steps = 8  # effective batch = batch_size * grad_accum_steps = 192 sequences * 512 tokens

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight

def precompute_rope(head_size, max_seq_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_size, 2).float() / head_size))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    # x: (B, n_head, T, head_size), interleaved-pair rotation
    x1, x2 = x[..., ::2], x[..., 1::2]
    cos = cos[None, None, :x.shape[2], :].to(x.dtype)
    sin = sin[None, None, :x.shape[2], :].to(x.dtype)
    rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated.flatten(-2)

class SwiGLU(nn.Module):
    def __init__(self, n_embd, hidden_mult=4):
        super().__init__()
        hidden = int(2 / 3 * hidden_mult * n_embd)
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(n_embd, hidden, bias=False)
        self.w2 = nn.Linear(hidden, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.n_head = n_head
        self.head_size = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout
        self.resid_dropout = nn.Dropout(dropout)
        cos, sin = precompute_rope(self.head_size, block_size)
        self.register_buffer('rope_cos', cos, persistent=False)
        self.register_buffer('rope_sin', sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        q = apply_rope(q, self.rope_cos, self.rope_sin)
        k = apply_rope(k, self.rope_cos, self.rope_sin)
        out = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.proj(out))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.sa = CausalSelfAttention(n_embd, n_head, block_size)
        self.ffwd = SwiGLU(n_embd)
        self.ln1 = RMSNorm(n_embd)
        self.ln2 = RMSNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding_table.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding_table(idx)  # RoPE encodes position inside attention, no learned pos embedding needed
        x = self.blocks(x)
        x = self.ln_f(x)

        if targets is None:
            logits = self.lm_head(x)
            loss = None
        else:
            logits = self.lm_head(x)
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
            # Avoid DataParallel gathering the full (B*T, vocab_size) logits tensor
            # back to GPU 0 every step -- with this vocab size that gather alone
            # allocates several GB and was the cause of a training-time OOM.
            logits = None

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

raw_model = GPTLanguageModel().to(device)
print(f'{sum(p.numel() for p in raw_model.parameters()) / 1e6:.2f}M parameters')

# --- Checkpoint resume: this kernel lists itself as a kernel_source in
# kernel-metadata.json, so the previous committed version's /kaggle/working
# output is mounted as input on the next run. This is what makes multi-session
# training across the weekly GPU quota possible instead of restarting from 0.
CHECKPOINT_OUT_PATH = '/kaggle/working/checkpoint.pt'
start_iter = 0
optimizer_state = None

resume_candidates = [p for p in glob.glob('/kaggle/input/*/checkpoint.pt')]
if resume_candidates:
    ckpt_path = resume_candidates[0]
    print(f'Found checkpoint at {ckpt_path}, attempting resume...')
    try:
        ckpt = torch.load(ckpt_path, map_location=device)
        raw_model.load_state_dict(ckpt['model_state_dict'])
        optimizer_state = ckpt['optimizer_state_dict']
        start_iter = ckpt['iter'] + 1
        print(f'Resumed from iter {ckpt["iter"]}, continuing at iter {start_iter}')
    except Exception as e:
        print('Checkpoint found but incompatible (likely a hyperparameter/architecture change), starting fresh:', e)
else:
    print('No checkpoint found, starting fresh')

model = raw_model
if device == 'cuda' and torch.cuda.device_count() > 1:
    print(f'Wrapping model in DataParallel across {torch.cuda.device_count()} GPUs')
    model = nn.DataParallel(raw_model)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
if optimizer_state is not None:
    optimizer.load_state_dict(optimizer_state)

scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

# WSD (warmup-stable-decay) schedule: stable plateau lets us stop/resume
# across weekly quota sessions without committing to a fixed total step count
# upfront the way a cosine schedule would.
warmup_iters = 150
decay_start_iter = int(max_iters * 0.8)
min_lr = learning_rate * 0.1

def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    if it < decay_start_iter:
        return learning_rate
    decay_ratio = (it - decay_start_iter) / max(1, (max_iters - decay_start_iter))
    decay_ratio = min(decay_ratio, 1.0)
    return learning_rate - decay_ratio * (learning_rate - min_lr)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                logits, loss = model(X, Y)
            losses[k] = loss.mean().item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
t_last = time.time()
tokens_since_last = 0

if start_iter >= max_iters:
    print(f'start_iter ({start_iter}) >= max_iters ({max_iters}), nothing left to train this run.')
else:
    for iter in range(start_iter, max_iters):
        lr = get_lr(iter)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        if iter % eval_interval == 0 or iter == max_iters - 1:
            losses = estimate_loss()
            elapsed = time.time() - t_last
            toks_per_sec = tokens_since_last / elapsed if elapsed > 0 else 0.0
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, "
                  f"lr {lr:.2e}, {toks_per_sec:,.0f} tok/s")
            t_last = time.time()
            tokens_since_last = 0

            torch.save({
                'model_state_dict': raw_model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'iter': iter,
            }, CHECKPOINT_OUT_PATH)

        optimizer.zero_grad(set_to_none=True)
        for micro_step in range(grad_accum_steps):
            xb, yb = get_batch('train')
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                logits, loss = model(xb, yb)
                loss = loss.mean() / grad_accum_steps
            scaler.scale(loss).backward()
            tokens_since_last += xb.numel()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

    print('final loss:', loss.item() * grad_accum_steps)

    torch.save({
        'model_state_dict': raw_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'iter': max_iters - 1,
    }, CHECKPOINT_OUT_PATH)

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = raw_model.generate(context, max_new_tokens=100)[0].tolist()
print(tokenizer.decode(generated))

## Beispiel-Vervollständigungen

Dieses Modell wird auf reinem `fineweb-edu`-Text vortrainiert, nicht auf Chat-Daten -- es ist noch ein Base-Modell, kein Assistent. Die Testfälle sind daher einfache Text-Fortsetzungen, kein Chat-Format. Instruction-Following kommt erst nach dem SFT-Schritt (Phase 4) auf `smol-smoltalk`.

In [ ]:
COMPLETION_MAX_NEW_TOKENS = 80
COMPLETION_TEMPERATURE = 0.7
COMPLETION_TOP_K = 50

def generate_completion(prompt, max_new_tokens=COMPLETION_MAX_NEW_TOKENS, temperature=COMPLETION_TEMPERATURE, top_k=COMPLETION_TOP_K):
    ids = tokenizer.encode(prompt)
    if len(ids) == 0:
        return '[prompt could not be tokenized]'

    context = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        output = raw_model.generate(context, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)

    generated_ids = output[0][len(ids):].tolist()
    return tokenizer.decode(generated_ids).strip()

# Plain continuation prompts, matching the fineweb-edu pretraining distribution
# (educational web text) -- no chat template, since this is still a base model.
example_prompts = [
    'The history of the Roman Empire begins with',
    'Photosynthesis is the process by which plants',
    'In machine learning, a neural network is',
    'The largest planet in our solar system is',
]

for prompt in example_prompts:
    completion = generate_completion(prompt)
    print('Prompt:', prompt)
    print('Completion:', completion)
    print('-' * 60)